In [0]:
# Section 1 — Read Bronze Table

df = spark.table("bike_lakehouse.bronze.erp_px_cat_g1v2")

display(df)

In [0]:
# Section 2 — Basic Quality Analysis

from pyspark.sql.functions import col

print("Total rows:", df.count())

print("\nSchema:")
df.printSchema()

print("\nNULL counts:")
for c in df.columns:
    print(c, df.filter(col(c).isNull()).count())

print("\nDistinct counts:")
for c in df.columns:
    print(c, df.select(c).distinct().count())

print("\nCategory values:")
display(
    df.groupBy("CAT")
      .count()
      .orderBy("CAT")
)

print("\nMaintenance values:")
display(
    df.groupBy("MAINTENANCE")
      .count()
      .orderBy("MAINTENANCE")
)

In [0]:
# Section 3 — Rename Columns

df_silver = (
    df
    .withColumnRenamed("ID", "category_id")
    .withColumnRenamed("CAT", "category")
    .withColumnRenamed("SUBCAT", "subcategory")
    .withColumnRenamed("MAINTENANCE", "maintenance_required")
)

df_silver.printSchema()

In [0]:
# Section 4 — Final Silver Sanity Check

print("Final row count:", df_silver.count())

print(
    "Duplicate category IDs:",
    df_silver
    .groupBy("category_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(
    "NULL category IDs:",
    df_silver.filter(col("category_id").isNull()).count()
)

print(
    "NULL categories:",
    df_silver.filter(col("category").isNull()).count()
)

print(
    "NULL subcategories:",
    df_silver.filter(col("subcategory").isNull()).count()
)

print(
    "NULL maintenance values:",
    df_silver.filter(col("maintenance_required").isNull()).count()
)

display(
    df_silver
    .groupBy("maintenance_required")
    .count()
    .orderBy("maintenance_required")
)

In [0]:
# Section 5 — Write Silver Table

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bike_lakehouse.silver.erp_product_categories")

In [0]:
df.display()

In [0]:
# Silver Layer Verification — Table Counts

silver_tables = [
    "bike_lakehouse.silver.crm_customers",
    "bike_lakehouse.silver.crm_products",
    "bike_lakehouse.silver.crm_sales_details",
    "bike_lakehouse.silver.erp_customers",
    "bike_lakehouse.silver.erp_locations",
    "bike_lakehouse.silver.erp_product_categories"
]

for table_name in silver_tables:
    count = spark.table(table_name).count()
    print(f"{table_name} → {count:,} rows")

In [0]:
# Silver Layer Verification — Schemas

for table_name in silver_tables:
    print(f"\n{'=' * 70}")
    print(table_name)
    print("=" * 70)
    spark.table(table_name).printSchema()

In [0]:
# Silver Layer Verification — Key & NULL Integrity

from pyspark.sql.functions import col

checks = {
    "crm_customers.customer_id": (
        "bike_lakehouse.silver.crm_customers",
        "customer_id"
    ),
    "crm_customers.customer_key": (
        "bike_lakehouse.silver.crm_customers",
        "customer_key"
    ),
    "crm_products.product_id": (
        "bike_lakehouse.silver.crm_products",
        "product_id"
    ),
    "crm_products.product_key": (
        "bike_lakehouse.silver.crm_products",
        "product_key"
    ),
    "crm_sales_details.order_number": (
        "bike_lakehouse.silver.crm_sales_details",
        "order_number"
    ),
    "crm_sales_details.product_key": (
        "bike_lakehouse.silver.crm_sales_details",
        "product_key"
    ),
    "crm_sales_details.customer_id": (
        "bike_lakehouse.silver.crm_sales_details",
        "customer_id"
    ),
    "erp_customers.customer_id": (
        "bike_lakehouse.silver.erp_customers",
        "customer_id"
    ),
    "erp_locations.customer_id": (
        "bike_lakehouse.silver.erp_locations",
        "customer_id"
    ),
    "erp_product_categories.category_id": (
        "bike_lakehouse.silver.erp_product_categories",
        "category_id"
    )
}

for name, (table_name, column_name) in checks.items():
    df_check = spark.table(table_name)

    null_count = df_check.filter(col(column_name).isNull()).count()

    duplicate_count = (
        df_check
        .groupBy(column_name)
        .count()
        .filter(col("count") > 1)
        .count()
    )

    print(
        f"{name:<45} "
        f"NULLs: {null_count:<5} "
        f"Duplicate values: {duplicate_count}"
    )

In [0]:
# Silver Layer Verification — Customer Cross-System Integrity

crm_customers = spark.table(
    "bike_lakehouse.silver.crm_customers"
)

erp_customers = spark.table(
    "bike_lakehouse.silver.erp_customers"
)

erp_locations = spark.table(
    "bike_lakehouse.silver.erp_locations"
)

print("CRM customer IDs:")
display(
    crm_customers
    .select("customer_id")
    .limit(10)
)

print("ERP customer IDs:")
display(
    erp_customers
    .select("customer_id")
    .limit(10)
)

print("ERP location customer IDs:")
display(
    erp_locations
    .select("customer_id")
    .limit(10)
)

In [0]:
# Silver Layer Verification — Corrected Customer ID Matching

from pyspark.sql.functions import col, regexp_extract

crm_ids = (
    crm_customers
    .select(
        col("customer_id").cast("string").alias("customer_id")
    )
    .distinct()
)

erp_customer_ids = (
    erp_customers
    .select(
        regexp_extract(
            col("customer_id"),
            r"(\d+)$",
            1
        ).cast("int").cast("string").alias("customer_id")
    )
    .distinct()
)

erp_location_ids = (
    erp_locations
    .select(
        regexp_extract(
            col("customer_id"),
            r"(\d+)$",
            1
        ).cast("int").cast("string").alias("customer_id")
    )
    .distinct()
)

crm_vs_erp = (
    crm_ids
    .join(erp_customer_ids, "customer_id", "left_anti")
    .count()
)

crm_vs_location = (
    crm_ids
    .join(erp_location_ids, "customer_id", "left_anti")
    .count()
)

print("CRM customers without ERP customer record:", crm_vs_erp)
print("CRM customers without ERP location record:", crm_vs_location)

In [0]:
# Silver Layer Verification — Sales Referential Integrity

sales = spark.table(
    "bike_lakehouse.silver.crm_sales_details"
)

customers = spark.table(
    "bike_lakehouse.silver.crm_customers"
)

products = spark.table(
    "bike_lakehouse.silver.crm_products"
)

# Customer integrity
sales_customer_ids = sales.select("customer_id").distinct()
crm_customer_ids = customers.select("customer_id").distinct()

unmatched_customers = (
    sales_customer_ids
    .join(crm_customer_ids, "customer_id", "left_anti")
    .count()
)

# Product integrity using the established suffix relationship
sales_product_keys = sales.select("product_key").distinct()

unmatched_products = (
    sales_product_keys.alias("s")
    .join(
        products.select("product_key").alias("p"),
        col("p.product_key").endswith(col("s.product_key")),
        "left_anti"
    )
    .count()
)

print("Sales customer IDs without CRM customer:", unmatched_customers)
print("Sales product keys without CRM product:", unmatched_products)